# 실습 1. Python 텍스트 데이터 처리 (Day 1 / 90분)

> 시나리오: **쇼핑몰 리뷰 데이터 정제** — `data/reviews.csv`
>
> Day 1(Pandas·정규식)은 이 저장소의 **로컬 노트북**으로 진행한다.

## Task
1. CSV 로드 후 **결측·이상치 리포트** 작성
2. `text` 컬럼 정제 — URL/특수문자/공백 정규화
3. **정규식**으로 광고 패턴(`[광고]`, `^협찬`, `구매처:`) 제거
4. `rating` 별 평균 텍스트 길이 / 단어 수 집계
5. 정제 결과를 `reviews_clean.parquet` 으로 저장

In [5]:
import re

text = "abc@gmail.com abc hello world 123-456-7890 ddd@exeample.com"

# 이메일 추출
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
print("이메일:", emails)

hello = re.findall(r"^abc", text)
print("hello:", hello)

이메일: ['abc@gmail.com', 'ddd@exeample.com']
hello: ['abc']


In [6]:
import pandas as pd

# data/reviews.csv 가 없으면: 터미널에서 `python data/make_reviews.py`
df = pd.read_csv("../data/reviews.csv")
print(df.shape)
df.head()

(10000, 5)


,id,user,text,rating,created_at
0,1,user087,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,2.0,2026/06/18 02:37
1,2,user072,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,4.0,23.02.2026
2,3,user021,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,2.0,2026-06-14T10:17:00
3,4,user045,[광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡,4.0,NaN
4,5,user047,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,5.0,2026-05-07


## 1) 결측·이상치 리포트

In [12]:
df.info()
print("\n결측치:\n", df.isna().sum())
print("\nrating 분포:\n", df["rating"].value_counts(dropna=False))

# created_at 은 형식이 제각각 → errors='coerce' 로 파싱 실패는 NaT 처리
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", format="mixed")
print("\n날짜 파싱 실패(NaT):", df["created_at"].isna().sum(), "건")
# TODO: NaT 행을 버릴지/보존할지 결정
df = df.dropna(subset=["created_at"])
df.info()
print("\ncreated_at dtype:", df["created_at"].dtype)


<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          8000 non-null   int64         
 1   user        8000 non-null   str           
 2   text        8000 non-null   str           
 3   rating      7596 non-null   float64       
 4   created_at  8000 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 1.0 MB

결측치:
 id              0
user            0
text            0
rating        404
created_at      0
dtype: int64

rating 분포:
 rating
4.0    1903
5.0    1850
1.0    1379
2.0    1328
3.0    1136
NaN     404
Name: count, dtype: int64

날짜 파싱 실패(NaT): 0 건
<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          8000 non-null   int64         
 1   user      

In [8]:
# NaT 행 확인: 어떤 created_at 값들이 파싱에 실패했는가?
nat_rows = df[df["created_at"].isna()]
print(f"NaT 행 개수: {len(nat_rows)}")
print("\n원본 created_at 값 샘플 (파싱 실패한 것들):")
print(nat_rows[["text", "created_at"]].head(20))

# 파싱 실패한 created_at 값들의 고유값 확인
print("\n\n파싱 실패한 created_at 원본 값 분포:")
print(nat_rows["text"].value_counts().head(10))

NaT 행 개수: 2000

원본 created_at 값 샘플 (파싱 실패한 것들):
                                                 text created_at
3        [광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡        NaT
8   협찬 받아 작성한 후기입니다. 재구매 의사 100% 강추합니다 🙏 https://e...        NaT
13               협찬 받아 작성한 후기입니다. 냄새가 너무 심해서 못 쓰겠어요 🔥        NaT
18          [광고] 지금 구매하면 사은품 증정! 배송이 일주일이나 걸렸습니다 별로 🔥        NaT
23   색감이 사진이랑 똑같아서 만족합니다 ㅠㅠ https://example.com/event        NaT
28                                 배송이 정말 빨라서 깜짝 놀랐어요        NaT
33                                  냄새가 너무 심해서 못 쓰겠어요        NaT
38  [광고] 지금 구매하면 사은품 증정! 배송이 일주일이나 걸렸습니다 별로 👍 http...        NaT
43   색감이 사진이랑 똑같아서 만족합니다 ㅠㅠ https://example.com/event        NaT
48           [광고] 지금 구매하면 사은품 증정! 재구매 의사 100% 강추합니다 🙏        NaT
53                                     가격대비 그럭저럭이에요 🙏        NaT
58         [광고] 지금 구매하면 사은품 증정! 고객센터 연결도 안 되고 최악이에요 👍        NaT
63                             한 번 쓰고 고장났습니다 환불 원해요 😍        NaT
68     사이즈도 딱 맞고 마감이 깔끔해요 🔥 https://exampl

In [9]:
# 원본 created_at 값을 보기 위해 다시 로드 (파싱 전)
df_raw = pd.read_csv("../data/reviews.csv")

# NaT인 인덱스들을 찾아서 원본 값 확인
nat_indices = df[df["created_at"].isna()].index
print("NaT가 된 행들의 원본 created_at 값 샘플:")
print(df_raw.loc[nat_indices, ["text", "created_at"]].head(20))

print("\n\n원본 created_at 값의 고유값 분포 (NaT가 된 것들만):")
print(df_raw.loc[nat_indices, "created_at"].value_counts().head(15))

NaT가 된 행들의 원본 created_at 값 샘플:
                                                 text created_at
3        [광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡        NaN
8   협찬 받아 작성한 후기입니다. 재구매 의사 100% 강추합니다 🙏 https://e...        NaN
13               협찬 받아 작성한 후기입니다. 냄새가 너무 심해서 못 쓰겠어요 🔥        NaN
18          [광고] 지금 구매하면 사은품 증정! 배송이 일주일이나 걸렸습니다 별로 🔥        NaN
23   색감이 사진이랑 똑같아서 만족합니다 ㅠㅠ https://example.com/event        NaN
28                                 배송이 정말 빨라서 깜짝 놀랐어요        NaN
33                                  냄새가 너무 심해서 못 쓰겠어요        NaN
38  [광고] 지금 구매하면 사은품 증정! 배송이 일주일이나 걸렸습니다 별로 👍 http...        NaN
43   색감이 사진이랑 똑같아서 만족합니다 ㅠㅠ https://example.com/event        NaN
48           [광고] 지금 구매하면 사은품 증정! 재구매 의사 100% 강추합니다 🙏        NaN
53                                     가격대비 그럭저럭이에요 🙏        NaN
58         [광고] 지금 구매하면 사은품 증정! 고객센터 연결도 안 되고 최악이에요 👍        NaN
63                             한 번 쓰고 고장났습니다 환불 원해요 😍        NaN
68     사이즈도 딱 맞고 마감이 깔끔해요 🔥 https://example.com/event      

In [10]:
# 확인: 원본 데이터에서 created_at이 NaN인 경우
print("원본 CSV에서 created_at의 결측치:")
print(df_raw["created_at"].isna().sum(), "건")
print("\n전체 행 개수:", len(df_raw))
print("created_at이 있는 행:", (~df_raw["created_at"].isna()).sum(), "건")

print("\n결론: NaT 2000건 = 원본 CSV에서 created_at이 missing된 행들")

원본 CSV에서 created_at의 결측치:
2000 건

전체 행 개수: 10000
created_at이 있는 행: 8000 건

결론: NaT 2000건 = 원본 CSV에서 created_at이 missing된 행들


## 2) 텍스트 정제 파이프라인 (슬라이드 `clean()` 참고)

`.str` 로 컬럼 전체를 한 번에 정제한다. 정규식 패턴 풀이:
- `https?://\S+` → http/https 로 시작하는 URL (공백 전까지)
- `[^\w가-힣\s]` → 단어문자(`\w`)·한글(`가-힣`)·공백(`\s`) **이 아닌** 것 = 특수문자/이모지
- `\s+` → 연속 공백을 1칸으로

In [13]:
import re

def clean(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .str.strip()
         .str.replace(r"https?://\S+", "", regex=True)        # URL 제거
         .str.replace(r"[^\w가-힣\s]", " ", regex=True)        # 특수문자 제거 (가-힣 = U+AC00~U+D7A3)
         .str.replace(r"\s+", " ", regex=True)                 # 공백 정규화
         .str.strip()
    )

df["clean_text"] = clean(df["text"])
df[["text", "clean_text"]].head()

,text,clean_text
0,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,협찬 받아 작성한 후기입니다 생각보다 너무 작고 부실해요
1,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,협찬 받아 작성한 후기입니다 배송이 정말 빨라서 깜짝 놀랐어요
2,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,배송이 일주일이나 걸렸습니다 별로
4,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,광고 지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요
5,[광고] 지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍,광고 지금 구매하면 사은품 증정 가성비 최고입니다 또 살게요


## 3) 광고 패턴 제거 (정규식)

In [15]:
AD = re.compile(r"\[광고\]|^협찬|구매처\s*:")
is_ad = df["text"].str.contains(AD)
print(f"광고성 리뷰: {is_ad.sum()}건")
# TODO: 광고 문구만 제거할지, 행 자체를 버릴지 결정하고 처리
df = df[~is_ad].copy()
print(f"광고성 리뷰 제거 후: {len(df)}건")
df.head()

광고성 리뷰: 3942건
광고성 리뷰 제거 후: 4058건


,id,user,text,rating,created_at,clean_text
2,3,user021,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,2.0,2026-06-14 10:17:00,배송이 일주일이나 걸렸습니다 별로
6,7,user069,가성비 최고입니다 또 살게요,5.0,2026-08-06 00:00:00,가성비 최고입니다 또 살게요
7,8,user005,사이즈도 딱 맞고 마감이 깔끔해요 🔥,4.0,2026-03-13 08:04:00,사이즈도 딱 맞고 마감이 깔끔해요
9,10,user029,재구매 의사 100% 강추합니다 😡,5.0,2026-02-17 00:00:00,재구매 의사 100 강추합니다
10,11,user050,생각보다 너무 작고 부실해요 😍 https://example.com/event,1.0,2026-04-20 14:33:00,생각보다 너무 작고 부실해요


## 4) rating 별 텍스트 길이 / 단어 수 집계

In [16]:
df["length"] = df["clean_text"].str.len()
df["words"] = df["clean_text"].str.split().str.len()
df.groupby("rating")[["length", "words"]].mean()

,length,words
rating,,
1.0,18.173977,4.837719
2.0,18.183644,4.819225
3.0,14.261986,3.945205
4.0,17.815401,4.497890
5.0,17.891693,4.503680


In [17]:
df.info()

<class 'pandas.DataFrame'>
Index: 4058 entries, 2 to 9999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          4058 non-null   int64         
 1   user        4058 non-null   str           
 2   text        4058 non-null   str           
 3   rating      3864 non-null   float64       
 4   created_at  4058 non-null   datetime64[us]
 5   clean_text  4058 non-null   str           
 6   length      4058 non-null   int64         
 7   words       4058 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 721.7 KB


## 5) 저장 — 실습 2의 입력이 된다

In [18]:
df.to_parquet("../data/reviews_clean.parquet", index=False)
print("saved: data/reviews_clean.parquet")

saved: data/reviews_clean.parquet
